# Chapter 9 — Escalation and Human-in-the-Loop · Practice

**Goal:** build every escalation and human-review control from the chapter with the Agents SDK and the Responses API: rule-based triggers, an escalate tool that ends the run, guardrails with tripwires, approval-gated tools, durable pause and resume with `RunState`, MCP approvals, a handoff summary filled from the run, threshold calibration and stratified audits. It ends with an approval-gated refund desk.

- Theory: [README.md](README.md) (sections 9.1–9.10). This notebook holds the code.
- **Estimated time:** 3–4 hours
- **Estimated API cost:** $0 for everything except the last cell; that one live run costs under $0.01 with the default `gpt-6-luna`.
- **Prerequisites:** [Chapter 3](../03-agents-sdk-and-agents-api/README.md) (agents, tools, guardrail basics), [Chapter 6](../06-prompt-engineering-and-evals/README.md) (labeled datasets), the repo venv from [00-prerequisites](../../00-prerequisites/README.md). `OPENAI_API_KEY` in `.env` only for the final live cell.

Every agent run here uses `agents.testing.ScriptedModel`, the SDK's own test model: you script which tool calls and messages "the model" returns, and the real runner does everything else (guardrails, approvals, serialization, resume). That makes approval mechanics fully reproducible without a key. Only the cell marked **(live)** calls the API.

In [ ]:
import asyncio
import html
import json
import math
import os
import random
import re
import uuid
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Any, Literal

import httpx
import openai
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from openai.types.responses import Response, ResponseInputItemParam, ToolParam
from openai.types.responses.response_output_item import McpApprovalRequest
from pydantic import BaseModel, Field, TypeAdapter

from agents import (
    Agent,
    AgentHooks,
    GuardrailFunctionOutput,
    HostedMCPTool,
    InputGuardrailTripwireTriggered,
    ItemHelpers,
    MaxTurnsExceeded,
    ModelSettings,
    OutputGuardrailTripwireTriggered,
    RunConfig,
    RunContextWrapper,
    RunErrorHandlerInput,
    RunErrorHandlerResult,
    Runner,
    RunState,
    StopAtTools,
    ToolErrorFormatterArgs,
    ToolExecutionConfig,
    ToolGuardrailFunctionOutput,
    function_tool,
    input_guardrail,
    output_guardrail,
    set_tracing_disabled,
    tool_input_guardrail,
)
from agents.mcp import MCPServerStreamableHttp
from agents.testing import ScriptedModel, assistant_message, function_call

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
set_tracing_disabled(True)  # offline demos; the live cell at the end turns tracing back on
print("Model for the live cell:", MODEL)

In [ ]:
# Only the final live cell needs a key. Every other cell runs without one.
if not os.getenv("OPENAI_API_KEY"):
    print("No OPENAI_API_KEY: every cell except the last (live) one still runs.")

Shared fixtures: a tiny order database, a payments log (so you can see whether money actually moved), a ticket queue for human escalations, and the run context. `customer_id` in the context comes from your authentication layer, never from the model. `show()` prints what happened in a run.

In [ ]:
def fresh_orders() -> dict[str, dict]:
    return {
        "A-1001": {"customer_id": "C-42", "item": "Phone case", "total_usd": 39.0, "status": "delivered", "refunded": False},
        "A-2002": {"customer_id": "C-42", "item": "Espresso machine", "total_usd": 640.0, "status": "delivered", "refunded": False},
        "A-3003": {"customer_id": "C-42", "item": "Headphones", "total_usd": 250.0, "status": "delivered", "refunded": True},
        "B-7007": {"customer_id": "C-77", "item": "Laptop stand", "total_usd": 120.0, "status": "delivered", "refunded": False},
    }


ORDERS: dict[str, dict] = fresh_orders()
PAYMENTS: list[tuple[str, float]] = []   # every refund that really executed
TICKETS: list[dict] = []                 # the human queue


def reset() -> None:
    ORDERS.clear()
    ORDERS.update(fresh_orders())
    PAYMENTS.clear()
    TICKETS.clear()


def create_ticket(payload: dict) -> str:
    TICKETS.append(payload)
    return f"T-{len(TICKETS):04d}"


@dataclass
class SupportCtx:
    customer_id: str          # set by your authentication, not by the model
    escalating: bool = False  # set by code when a model response contains escalate_to_human (9.3)


def owned_order(ctx: Any, order_id: Any) -> dict | None:
    """The order, but only if it belongs to the authenticated customer. No customer -> None (fail closed)."""
    order = ORDERS.get(order_id) if isinstance(order_id, str) else None
    customer_id = getattr(ctx, "customer_id", None)
    return order if order is not None and customer_id and order["customer_id"] == customer_id else None


def _name(raw: Any) -> str | None:
    return raw.get("name") if isinstance(raw, dict) else getattr(raw, "name", None)


def show(result, label: str = "") -> None:
    """Print the items of a run, any pending approvals, and the final output."""
    print(f"--- {label}")
    for item in result.new_items:
        if item.type == "tool_call_item":
            print(f"  tool call    {_name(item.raw_item)}({getattr(item.raw_item, 'arguments', '')})")
        elif item.type == "tool_call_output_item":
            print(f"  tool output  {str(item.output)[:110]}")
        elif item.type == "message_output_item":
            print(f"  message      {ItemHelpers.text_message_output(item)[:110]}")
        else:
            print(f"  {item.type}")
    for pending in result.interruptions:
        print(f"  PAUSED       {pending.name}({pending.arguments}) call_id={pending.call_id}")
    print(f"  final_output {result.final_output!r}")


def seen_by_model(model: ScriptedModel) -> list[str]:
    """Tool results the model received on its most recent call."""
    items = model.last_call.input if model.last_call else []
    return [x["output"] for x in items if isinstance(x, dict) and x.get("type") == "function_call_output"]

## 9.1 When to escalate

Good escalation triggers are **observable conditions**: an explicit request for a person, a request outside policy, a failure counter, a high-risk action, an ambiguous identity. Sentiment and the model's self-rated confidence look attractive but misroute. Rate each tool's risk before writing prompts. [Theory](README.md#91-when-to-escalate)

Below is a labeled set of customer situations. Each case records what your systems can observe (topic from intent detection, clarifications asked so far, how many customers a lookup matched, the refund amount) plus two weak signals: a sentiment score and a self-rated confidence. Those last two numbers are **illustrative**, not real model output.

In [ ]:
COVERED_TOPICS = {"late_delivery", "damaged", "refund", "address_change", "duplicate_charge", "order_status"}
AUTO_LIMIT_USD = 100    # above this, a human approves (9.4)
HARD_LIMIT_USD = 500    # above this, the agent may not refund at all (9.3)

# "asks to talk to / wants a human-like role" -- a verb of asking must come first
HUMAN_REQUEST = re.compile(
    r"\b(want|need|talk to|speak (?:to|with)|get me|connect me|transfer me|put me through)\b"
    r"[^.?!]{0,25}\b(human|person|manager|supervisor)\b", re.I)


@dataclass
class Case:
    text: str
    expected: str
    topic: str = "refund"
    clarifications: int = 0
    matches: int = 1
    refund_usd: float = 0.0
    sentiment: float = 0.0       # -1 (angry) .. +1 (happy), from a sentiment model (illustrative)
    self_confidence: int = 8     # model's answer to "rate your confidence 1-10" (illustrative)


CASES = [
    Case("This is ridiculous, I'm so frustrated! My parcel is 3 days late.", "resolve", "late_delivery", sentiment=-0.9, self_confidence=9),
    Case("I want to talk to a human.", "escalate:customer_requested_human", sentiment=-0.2, self_confidence=9),
    Case("Another shop offers free next-day delivery. Give me that too.", "escalate:policy_gap", "competitor_offer", sentiment=0.1, self_confidence=8),
    Case("My manager bought this for me, can I return it?", "resolve", "refund", refund_usd=39, sentiment=0.3, self_confidence=9),
    Case("The espresso machine arrived broken, refund all $640.", "escalate:policy_limit", "damaged", refund_usd=640, sentiment=-0.4, self_confidence=9),
    Case("Please refund the $250 headphones.", "approval", "refund", refund_usd=250, sentiment=0.2, self_confidence=9),
    Case("Change the address on my order, I'm Maria Lopez.", "ask_identifier", "address_change", matches=3, sentiment=0.0, self_confidence=7),
    Case("The thing... you know... just fix it.", "escalate:cannot_progress", "refund", clarifications=2, sentiment=-0.1, self_confidence=6),
    Case("Thanks! Where is order A-1001?", "resolve", "order_status", sentiment=0.8, self_confidence=10),
    Case("You charged me twice, worst service ever!!!", "resolve", "duplicate_charge", refund_usd=39, sentiment=-1.0, self_confidence=5),
    Case("Our lawyers will contact you about the data you leaked.", "escalate:policy_gap", "legal_threat", sentiment=-0.1, self_confidence=9),
    Case("Can I speak with a supervisor, please?", "escalate:customer_requested_human", sentiment=0.3, self_confidence=10),
]


def route_by_rules(c: Case) -> str:
    """Observable triggers, in priority order."""
    if HUMAN_REQUEST.search(c.text):
        return "escalate:customer_requested_human"
    if c.matches > 1:
        return "ask_identifier"                       # never guess between customers
    if c.topic not in COVERED_TOPICS:
        return "escalate:policy_gap"
    if c.clarifications >= 2:
        return "escalate:cannot_progress"
    if c.refund_usd > HARD_LIMIT_USD:
        return "escalate:policy_limit"
    if c.refund_usd > AUTO_LIMIT_USD:
        return "approval"
    return "resolve"


def route_by_sentiment(c: Case) -> str:
    return "escalate:angry" if c.sentiment < -0.5 else "resolve"


def route_by_self_confidence(c: Case) -> str:
    return "escalate:low_confidence" if c.self_confidence < 7 else "resolve"


def to_human(route: str) -> bool:
    return route.startswith("escalate")


for name, router in [("rules", route_by_rules), ("sentiment", route_by_sentiment), ("self-confidence", route_by_self_confidence)]:
    exact = sum(router(c) == c.expected for c in CASES)
    human_ok = sum(to_human(router(c)) == to_human(c.expected) for c in CASES)
    print(f"{name:<16} exact route {exact:>2}/{len(CASES)}   human-or-not {human_ok:>2}/{len(CASES)}")
    for c in CASES:
        if to_human(router(c)) != to_human(c.expected):
            print(f"    misroute: {c.text[:55]!r:<58} -> {router(c)} (expected {c.expected})")

> **Observe:** the sentiment router escalates the angry-but-simple cases (late parcel, duplicate charge) and lets the calm legal threat and the competitor's delivery offer through. The self-confidence router sends a duplicate charge to a human and keeps a confident policy gap. The rules router gets every case right because each rule checks something observable. Also note that "My manager bought this" does **not** trigger: the regex wants a verb of asking first.
>
> **Try it:** add `Case("Put me through to your boss.", "escalate:customer_requested_human")`. The regex misses it. That is why 9.3 measures the detector on labeled real messages before trusting it.

Next, the tool risk rating from OpenAI's practical guide: read vs write, reversibility, financial impact. Do this in the solution plan, before the prompt exists.

In [ ]:
TOOL_PROFILES = [
    {"tool": "lookup_order", "write": False, "reversible": True, "financial": "none"},
    {"tool": "update_address", "write": True, "reversible": True, "financial": "low"},
    {"tool": "issue_refund <= $100", "write": True, "reversible": False, "financial": "low"},
    {"tool": "issue_refund > $100", "write": True, "reversible": False, "financial": "medium"},
    {"tool": "issue_refund > $500", "write": True, "reversible": False, "financial": "high"},
]
CONTROLS = {
    "Low": "ownership check in the tool",
    "Medium": "deterministic check (ownership, policy gate)",
    "High": "policy gate + human approval",
    "Blocked": "gate rejects; agent must escalate",
}


def risk_rating(p: dict) -> str:
    if not p["write"]:
        return "Low"
    if p["financial"] == "high" and not p["reversible"]:
        return "Blocked"
    if p["financial"] == "medium" or (p["financial"] == "high" and p["reversible"]):
        return "High"
    return "Medium"


print(f"{'tool':<22} {'rating':<8} control")
for p in TOOL_PROFILES:
    r = risk_rating(p)
    print(f"{p['tool']:<22} {r:<8} {CONTROLS[r]}")

## 9.2 Escalation patterns and the escalate tool

Escalation is a strict tool the model can call. `StopAtTools` ends the run after the turn in which it fires: the model gets **no further call**, so it can't keep chatting after handing off. It does not cancel other tool calls in the **same** model response; those still run (9.3 shows the fix). `error_handlers={"max_turns": ...}` turns "ran out of turns" into an escalation instead of an exception. [Theory](README.md#92-escalation-patterns-and-the-escalate-tool)

In [ ]:
class HumanHandoff(BaseModel):
    customer_id: str
    order_id: str | None                          # null if no order is involved
    escalation_reason: Literal["customer_requested_human", "policy_gap", "policy_limit",
                               "cannot_progress", "identity_ambiguous"]
    issue_summary: str = Field(description="2-3 sentences a human can act on without the transcript")
    customer_request: str = Field(description="What the customer wants, in their terms")
    actions_taken: list[str]
    amount_usd: float | None
    recommended_action: str
    customer_sentiment_note: str | None           # context for the human, NOT a routing signal


@function_tool
def lookup_order(ctx: RunContextWrapper[SupportCtx], order_id: str) -> str:
    """Look up one of the customer's orders by ID. Returns status, item and total in USD."""
    order = owned_order(ctx.context, order_id)     # ownership from the authenticated context
    if order is None:
        return json.dumps({"error": "not_found"})  # same answer for "missing" and "someone else's": no leak
    return json.dumps({k: order[k] for k in ("status", "item", "total_usd")})


@function_tool
def escalate_to_human(handoff: HumanHandoff) -> str:
    """Hand the case to a human agent. Use when the customer asks for a person, the request is not
    covered by policy, a refund needs approval you cannot get, or you cannot make progress."""
    return json.dumps({"ticket_id": create_ticket(handoff.model_dump())})


schema = escalate_to_human.params_json_schema["$defs"]["HumanHandoff"]
print("strict tool:", escalate_to_human.strict_json_schema)
print("required:", schema["required"])
print("additionalProperties:", schema["additionalProperties"])
print("nullable:", [k for k, v in schema["properties"].items() if {"type": "null"} in v.get("anyOf", [])])

In [ ]:
ESCALATION_INSTRUCTIONS = """# Escalation
Call escalate_to_human, and do not try to solve the issue first, when:
* the customer explicitly asks for a human, manager or supervisor;
* the request is not covered by the refund policy (for example matching another shop's offer);
* a tool tells you a request needs a human.
If the customer is upset but has not asked for a person, acknowledge it and offer a concrete fix first.
If a customer lookup returns more than one match, ask for the order ID or email. Never pick one."""

MANAGER_HANDOFF = {"handoff": {
    "customer_id": "C-42", "order_id": None, "escalation_reason": "customer_requested_human",
    "issue_summary": "Customer asked for a manager on the first message. No issue details yet.",
    "customer_request": "Talk to a manager", "actions_taken": [], "amount_usd": None,
    "recommended_action": "Call the customer back", "customer_sentiment_note": None}}


def support_agent(model: ScriptedModel, stop_on_escalation: bool = True) -> Agent:
    return Agent[SupportCtx](
        name="Support agent", instructions=ESCALATION_INSTRUCTIONS, model=model,
        tools=[lookup_order, escalate_to_human],
        model_settings=ModelSettings(parallel_tool_calls=False),   # at most one tool call per response (API setting)
        tool_use_behavior=StopAtTools(stop_at_tool_names=["escalate_to_human"]) if stop_on_escalation else "run_llm_again",
    )


# Pattern 1: immediate escalation, with StopAtTools
reset()
model = ScriptedModel([[function_call("escalate_to_human", MANAGER_HANDOFF, call_id="esc_1")]])
result = await Runner.run(support_agent(model), "Get me a manager.", context=SupportCtx("C-42"))
show(result, "with StopAtTools")
print("model calls:", len(model.calls), "| tickets:", len(TICKETS))

# The same without StopAtTools: the model gets another turn after handing off
reset()
model = ScriptedModel([
    [function_call("escalate_to_human", MANAGER_HANDOFF, call_id="esc_1")],
    [assistant_message("I've escalated this. Meanwhile, let me see if I can fix it myself...")],
])
result = await Runner.run(support_agent(model, stop_on_escalation=False), "Get me a manager.", context=SupportCtx("C-42"))
show(result, "without StopAtTools")
print("model calls:", len(model.calls))

> **Observe:** with `StopAtTools`, `final_output` is the escalate tool's return value (the ticket JSON) and there is exactly one model call. Without it, the model gets another turn and can undercut the handoff ("let me fix it myself").
>
> `ModelSettings(parallel_tool_calls=False)` asks the API for at most one tool call per response. The function calling guide says this "ensures exactly zero or one tool is called". `ScriptedModel` ignores that setting, so 9.3 also adds a check in code.

Pattern 3, **acknowledge → resolve → escalate on reiteration**, spans two user turns. The first angry message gets an acknowledgement and a concrete offer; only the explicit "Put me through to a person" escalates.

In [ ]:
reset()
model = ScriptedModel([
    [function_call("lookup_order", {"order_id": "A-1001"}, call_id="look_1")],
    [assistant_message("Sorry about that. A new case can ship today, or I can refund the $39. Which works for you?")],
    [function_call("escalate_to_human", {"handoff": {
        "customer_id": "C-42", "order_id": "A-1001", "escalation_reason": "customer_requested_human",
        "issue_summary": "Phone case A-1001 is faulty. Customer declined replacement and refund and asked for a person.",
        "customer_request": "Speak to a person", "actions_taken": ["Offered replacement or $39 refund"],
        "amount_usd": None, "recommended_action": "Call back; ask what outcome they expect",
        "customer_sentiment_note": "Frustrated"}}, call_id="esc_1")],
])
agent = support_agent(model)
ctx = SupportCtx("C-42")
turn1 = await Runner.run(agent, "My phone case A-1001 cracked on day two. Seriously?", context=ctx)
show(turn1, "turn 1: acknowledge and offer a fix")
turn2 = await Runner.run(agent, turn1.to_input_list() + [{"role": "user", "content": "No thanks. Put me through to a person."}], context=ctx)
show(turn2, "turn 2: explicit request -> escalate")

# new_items holds only what THIS run generated. Turn 1's lookup is history input to turn 2, not a new item.
print("\ntool calls in turn2.new_items:", [_name(i.raw_item) for i in turn2.new_items if i.type == "tool_call_item"])
# So keep a server-side action ledger per conversation and append each run's facts to it.
LEDGER = [_name(i.raw_item) for r in (turn1, turn2) for i in r.new_items if i.type == "tool_call_item"]
print("conversation ledger:            ", LEDGER)

**Failure thresholds.** `max_turns` is a safety cap. Without a handler, hitting it raises `MaxTurnsExceeded`. With `error_handlers={"max_turns": fn}`, your function receives the run data (`new_items`, `history`, `last_agent`) and returns a controlled final output: here, a ticket with reason `cannot_progress`.

In [ ]:
def _field(raw: Any, key: str) -> Any:
    return raw.get(key) if isinstance(raw, dict) else getattr(raw, key, None)


def actions_from_run(new_items) -> list[str]:
    """Facts for actions_taken, taken from what actually ran (dissected in 9.7)."""
    actions, escalation_calls = [], set()
    for item in new_items:
        if item.type == "tool_call_item":
            raw = item.raw_item
            if _field(raw, "type") != "function_call":       # hosted/built-in calls (web search, MCP...) have no arguments
                actions.append(f"Used {_field(raw, 'type') or 'a hosted tool'}")
            elif _name(raw) == "escalate_to_human":
                escalation_calls.add(_field(raw, "call_id"))   # the handoff itself is not an action taken
            else:
                actions.append(f"Called {_name(raw)}({_field(raw, 'arguments')})")
        elif item.type == "tool_call_output_item" and _field(item.raw_item, "call_id") not in escalation_calls:
            actions.append(f"  -> {str(item.output)[:200]}")
    return actions


def escalate_on_max_turns(data: RunErrorHandlerInput) -> RunErrorHandlerResult:
    ctx: SupportCtx = data.context.context
    ticket_id = create_ticket({
        "customer_id": ctx.customer_id, "escalation_reason": "cannot_progress",
        "actions_taken": actions_from_run(data.run_data.new_items),
        "last_agent": data.run_data.last_agent.name,
    })
    return RunErrorHandlerResult(final_output={"route": "human", "ticket_id": ticket_id}, include_in_history=False)


def looping_model() -> ScriptedModel:   # a model that keeps looking things up and never answers
    return ScriptedModel([[function_call("lookup_order", {"order_id": "A-9999"}, call_id=f"look_{i}")] for i in range(5)])


reset()
try:
    await Runner.run(support_agent(looping_model()), "Where is my stuff?", context=SupportCtx("C-42"), max_turns=3)
except MaxTurnsExceeded as exc:
    print("Without a handler:", type(exc).__name__)

result = await Runner.run(support_agent(looping_model()), "Where is my stuff?", context=SupportCtx("C-42"),
                          max_turns=3, error_handlers={"max_turns": escalate_on_max_turns})
print("With a handler:   ", result.final_output)
print(json.dumps(TICKETS[-1], indent=2))

> **Try it:** the same `error_handlers` dict accepts `"model_refusal"` and `"invalid_final_output"`. Add a handler for one of them and think about which reason code it should map to.

## 9.3 Enforcement vs prompt guidance: guardrails and policy gates

Instructions are probabilistic. Money, identity and compliance rules go into code: a **tool input guardrail** sees the raw arguments plus your context and can `reject_content(msg)` (the tool does not run and the model reads `msg`), input and output guardrails trip on the user message or the final answer, and a check inside the tool is the last line of defense. [Theory](README.md#93-enforcement-vs-prompt-guidance-guardrails-and-policy-gates)

In [ ]:
def valid_amount(amount: Any) -> bool:
    """A refund amount must be a real, finite, positive number. bool is not a number; NaN and infinity are not amounts."""
    return (isinstance(amount, (int, float)) and not isinstance(amount, bool)
            and math.isfinite(amount) and amount > 0)


@tool_input_guardrail
def refund_policy_gate(data) -> ToolGuardrailFunctionOutput:
    ctx = data.context.context
    if getattr(ctx, "escalating", False):     # the same model response also escalated: the human decides
        return ToolGuardrailFunctionOutput.reject_content("You are handing this case to a human. Do not refund now.")
    try:
        args = json.loads(data.context.tool_arguments or "{}")
    except json.JSONDecodeError:
        args = None
    if not isinstance(args, dict):
        return ToolGuardrailFunctionOutput.reject_content("Malformed refund request. Call issue_refund again with valid arguments.")
    order = owned_order(ctx, args.get("order_id"))
    amount = args.get("amount_usd")
    if order is None:
        return ToolGuardrailFunctionOutput.reject_content("Order not found for this customer. Ask for the order ID again.")
    if not valid_amount(amount):
        return ToolGuardrailFunctionOutput.reject_content("The refund amount must be a positive number. Ask the customer for the amount.")
    if order["refunded"]:
        return ToolGuardrailFunctionOutput.reject_content("This order was already refunded. Do not refund it again.")
    if amount > order["total_usd"] or amount > HARD_LIMIT_USD:
        return ToolGuardrailFunctionOutput.reject_content(
            f"Refunds above the order total or ${HARD_LIMIT_USD} need a human. "
            "Call escalate_to_human with escalation_reason='policy_limit'.")
    return ToolGuardrailFunctionOutput.allow()


def _refund(ctx: RunContextWrapper[SupportCtx], order_id: str, amount_usd: float) -> str:
    """Refund one of the customer's orders. Amount in USD, more than 0 and at most the order total."""
    order = owned_order(ctx.context, order_id)   # last line of defense: the same checks as the gate, again
    if (order is None or not valid_amount(amount_usd) or order["refunded"]
            or amount_usd > min(order["total_usd"], HARD_LIMIT_USD)):
        return json.dumps({"error": "refund_not_allowed"})   # an approval never makes an invalid refund valid
    order["refunded"] = True
    PAYMENTS.append((order_id, amount_usd))
    return json.dumps({"status": "refunded", "order_id": order_id, "amount_usd": amount_usd})


issue_refund_gated = function_tool(_refund, name_override="issue_refund", tool_input_guardrails=[refund_policy_gate])

GATE_CASES = [("B-7007", 120.0, "someone else's order"), ("A-3003", 50.0, "already refunded"),
              ("A-2002", 640.0, "above the hard limit"), ("A-1001", -50.0, "negative amount"),
              ("A-1001", 39.0, "within policy")]
for order_id, amount, label in GATE_CASES:
    reset()
    model = ScriptedModel([[function_call("issue_refund", {"order_id": order_id, "amount_usd": amount}, call_id="r1")],
                           [assistant_message("(model reacts to the tool result)")]])
    agent = Agent[SupportCtx](name="Refunds", model=model, tools=[issue_refund_gated])
    await Runner.run(agent, "refund please", context=SupportCtx(customer_id="C-42"))
    print(f"{label:<22} model saw: {seen_by_model(model)[0][:75]:<77} payments: {PAYMENTS}")

> **Observe:** four refunds never executed, and each time the model got a message it can act on ("ask again", "don't refund twice", "escalate", "ask for the amount"). Without the amount check, the $-50 "refund" would have passed every upper-bound rule. `reject_content` is the OpenAI counterpart of a Claude `PreToolUse` hook returning `deny` with a reason. `lookup_order` checks ownership too: for `B-7007` it answers `not_found`, the same as for an order that doesn't exist.

**Escalation and a side effect in the same response.** `StopAtTools` stops the run *after* the turn's tool calls have executed. If one model response contains both `escalate_to_human` and `issue_refund`, the refund still runs. `parallel_tool_calls=False` prevents such a response on the API side. An `AgentHooks.on_llm_end` hook makes it deterministic in code: it runs after the model responds and before any tool executes, so it can flag the escalation, and the policy gate then rejects every side effect in that response.

In [ ]:
class EscalationWins(AgentHooks):
    """If a model response contains escalate_to_human, flag it before any tool in that response runs."""
    async def on_llm_end(self, context, agent, response) -> None:
        context.context.escalating = any(_name(i) == "escalate_to_human" for i in response.output)


mixed_response = [function_call("escalate_to_human", MANAGER_HANDOFF, call_id="esc_1"),
                  function_call("issue_refund", {"order_id": "A-1001", "amount_usd": 39.0}, call_id="r1")]
for label, hooks in [("StopAtTools only", None), ("+ EscalationWins hook", EscalationWins())]:
    reset()
    agent = Agent[SupportCtx](name="Support agent", model=ScriptedModel([mixed_response]), hooks=hooks,
                              tools=[issue_refund_gated, escalate_to_human],
                              model_settings=ModelSettings(parallel_tool_calls=False),   # ScriptedModel ignores it
                              tool_use_behavior=StopAtTools(stop_at_tool_names=["escalate_to_human"]))
    result = await Runner.run(agent, "Refund A-1001. Actually, get me a manager.", context=SupportCtx("C-42"))
    print(f"{label:<22} final: {result.final_output}  tickets: {len(TICKETS)}  payments: {PAYMENTS}")

**Explicit human request as a blocking input guardrail.** With `run_in_parallel=False`, the guardrail finishes before the agent starts, so a tripwire costs no model tokens and no tool calls.

In [ ]:
@input_guardrail(run_in_parallel=False)
async def explicit_human_request(ctx, agent, user_input) -> GuardrailFunctionOutput:
    text = user_input if isinstance(user_input, str) else json.dumps(user_input)
    hit = bool(HUMAN_REQUEST.search(text))
    return GuardrailFunctionOutput(output_info={"trigger": "customer_requested_human" if hit else None},
                                   tripwire_triggered=hit)


model = ScriptedModel([[assistant_message("never used")]])
guarded = Agent[SupportCtx](name="Support agent", model=model, input_guardrails=[explicit_human_request])
try:
    await Runner.run(guarded, "Can I speak with a supervisor?", context=SupportCtx("C-42"))
except InputGuardrailTripwireTriggered as exc:
    reason = exc.guardrail_result.output.output_info["trigger"]
    print("Tripwire ->", reason, "| model calls:", len(model.calls))

# A regex is cheap but naive. Measure it on labeled messages before trusting it.
LABELED = [
    ("I want to talk to a human", True), ("Can I speak with a real person?", True),
    ("Get me your manager now", True), ("Connect me to a supervisor please", True),
    ("Put me through to your boss", True), ("Is there anyone I can call?", True),
    ("My manager bought this for me", False), ("A human error caused the delay?", False),
    ("Where is my parcel?", False), ("I need a refund for the person who ordered it", False),
]
tp = sum(bool(HUMAN_REQUEST.search(t)) and y for t, y in LABELED)
fp = sum(bool(HUMAN_REQUEST.search(t)) and not y for t, y in LABELED)
fn = sum((not HUMAN_REQUEST.search(t)) and y for t, y in LABELED)
print(f"precision {tp / max(tp + fp, 1):.2f}  recall {tp / max(tp + fn, 1):.2f}")
for t, y in LABELED:
    if bool(HUMAN_REQUEST.search(t)) != y:
        print("  wrong on:", t)

> **Observe:** the regex is right most of the time and still wrong in both directions: it misses "your boss" and "anyone I can call", and it fires on "I need a refund for the person who ordered it". A false positive sends a simple refund to the human queue; a miss makes the customer ask twice. OpenAI's guardrail docs show the alternative: a small guardrail agent with a structured `output_type`. Measure either one on labeled real messages the same way, and keep the `escalate_to_human` tool as the second line.

**Output guardrail.** It runs on the final answer of the agent that produces it. Use it to stop promises the agent is not allowed to make.

In [ ]:
PROMISE = re.compile(r"\b(refund (?:is|has been) approved|we will refund|guaranteed refund)\b", re.I)


@output_guardrail
async def no_unapproved_promises(ctx, agent, output) -> GuardrailFunctionOutput:
    hit = bool(PROMISE.search(str(output)))
    return GuardrailFunctionOutput(output_info={"reason": "unapproved_promise" if hit else None}, tripwire_triggered=hit)


model = ScriptedModel([[assistant_message("Good news: your refund is approved! Expect $640 tomorrow.")]])
agent = Agent(name="Support agent", model=model, output_guardrails=[no_unapproved_promises])
try:
    await Runner.run(agent, "Can I get my money back?")
except OutputGuardrailTripwireTriggered as exc:
    print("Blocked final answer:", exc.guardrail_result.output.output_info)
    print("Send instead: 'A specialist is reviewing your request and will reply within 24 hours.'")

**Workflow boundaries.** Input guardrails run only on the **first** agent. Here the explicit-request guardrail sits on a specialist reached through a handoff, so it never runs.

In [ ]:
specialist = Agent(name="Refund specialist", model=ScriptedModel([[assistant_message("Let me solve that for you!")]]),
                   input_guardrails=[explicit_human_request])
triage = Agent(name="Triage", handoffs=[specialist],
               model=ScriptedModel([[function_call("transfer_to_refund_specialist", {}, call_id="h1")]]))
result = await Runner.run(triage, "I want to talk to a manager about my refund.")
print("last agent:", result.last_agent.name, "| final:", result.final_output)
print("guardrail results:", result.input_guardrail_results)

> **Observe:** no tripwire, empty guardrail results, and the specialist happily "solves" a case where the customer asked for a person. Attach the guardrail to the first agent (or pass it in `RunConfig(input_guardrails=[...])`), and put side-effect checks on the tool that has the side effect.
>
> **Try it:** move `explicit_human_request` to `triage` and re-run. You need a fresh `ScriptedModel` for each run because a script is consumed as it runs.

## 9.4 Approval-gated tools in the Agents SDK

`needs_approval` (a bool, or a per-call callable `(ctx, params, call_id) -> bool`) pauses the run before the tool executes. `Runner.run` **returns** with `final_output=None` and `result.interruptions`; you `approve` or `reject` on `result.to_state()` and resume the **same** run with `Runner.run(agent, state)`. [Theory](README.md#94-approval-gated-tools-in-the-agents-sdk)

In [ ]:
POLICY_CALLS: list[dict] = []   # so you can see when the SDK consults your policy


async def needs_human(ctx: RunContextWrapper[SupportCtx], params: dict, call_id: str) -> bool:
    POLICY_CALLS.append(params)
    return float(params.get("amount_usd", 0)) > AUTO_LIMIT_USD


issue_refund = function_tool(_refund, name_override="issue_refund",
                             needs_approval=needs_human, tool_input_guardrails=[refund_policy_gate])


def refund_agent(model: ScriptedModel) -> Agent:
    return Agent[SupportCtx](name="Refund desk", model=model, tools=[lookup_order, issue_refund])


reset()
model = ScriptedModel([
    [function_call("issue_refund", {"order_id": "A-2002", "amount_usd": 300.0}, call_id="r1")],
    [assistant_message("Your $300 refund for the espresso machine is on its way.")],
])
agent = refund_agent(model)
result = await Runner.run(agent, "Refund $300 on A-2002, it arrived damaged.", context=SupportCtx("C-42"))
show(result, "paused")
item = result.interruptions[0]
print("item:", type(item).__name__, "| name:", item.name, "| call_id:", item.call_id, "| agent:", item.agent.name)
print("payments so far:", PAYMENTS)

state = result.to_state()
state.approve(item)
resumed = await Runner.run(agent, state)
show(resumed, "approved and resumed")
print("payments:", PAYMENTS)

**Rejecting with a message.** `rejection_message` is the exact text the model reads in place of the tool result. A run-wide default comes from `RunConfig(tool_error_formatter=...)` (check `args.kind == "approval_rejected"`); the per-call message wins.

In [ ]:
def run_formatter(args: ToolErrorFormatterArgs) -> str | None:
    if args.kind == "approval_rejected":
        return json.dumps({"error": "rejected_by_reviewer", "next_step": "Offer to escalate to a human."})
    return None   # keep the SDK default for other error kinds


for label, message in [("per-call message", "A supervisor declined this refund. Offer store credit or escalate."),
                       ("run-wide formatter", None)]:
    reset()
    model = ScriptedModel([
        [function_call("issue_refund", {"order_id": "A-2002", "amount_usd": 300.0}, call_id="r1")],
        [assistant_message("(model explains the outcome)")],
    ])
    agent = refund_agent(model)
    config = RunConfig(tool_error_formatter=run_formatter)
    result = await Runner.run(agent, "Refund $300 on A-2002.", context=SupportCtx("C-42"), run_config=config)
    state = result.to_state()
    state.reject(result.interruptions[0], rejection_message=message)
    await Runner.run(agent, state, run_config=config)
    print(f"{label:<19} model saw: {seen_by_model(model)[-1]}   payments: {PAYMENTS}")

**A callable policy fails closed.** If the SDK can't safely inspect the arguments (malformed JSON, not an object, `NaN` / `Infinity`), it **skips your callable** and requires manual approval. Watch `POLICY_CALLS`: it shows whether your policy was consulted.

In [ ]:
ARG_VARIANTS = [
    ('{"order_id": "A-1001", "amount_usd": 39.0}', "well-formed, small amount"),
    ('{"order_id": "A-1001", "amount_usd": NaN}', "NaN constant"),
    ('["A-1001", 39.0]', "valid JSON, not an object"),
    ('{"order_id": "A-1001", "amount_usd": 39', "malformed JSON"),
    ('{"order_id": "A-1001", "amount_usd": 39}', "integer for a float parameter"),
]
for raw_args, label in ARG_VARIANTS:
    reset()
    POLICY_CALLS.clear()
    model = ScriptedModel([[function_call("issue_refund", raw_args, call_id="r1")], [assistant_message("done")]])
    result = await Runner.run(refund_agent(model), "refund", context=SupportCtx("C-42"))
    outcome = "PAUSED for a human" if result.interruptions else "ran automatically"
    print(f"{label:<32} policy consulted: {bool(POLICY_CALLS)!s:<5}  -> {outcome}")

# Fail-closed means "ask a human", not "allow". A reviewer who approves the NaN call still can't move money:
# by default the gate runs after approval, and _refund repeats the amount check.
reset()
model = ScriptedModel([[function_call("issue_refund", '{"order_id": "A-1001", "amount_usd": NaN}', call_id="r1")],
                       [assistant_message("(model reacts)")]])
agent = refund_agent(model)
result = await Runner.run(agent, "refund", context=SupportCtx("C-42"))
state = result.to_state()
state.approve(result.interruptions[0])
await Runner.run(agent, state)
print(f"\nNaN call approved by a reviewer -> model saw: {seen_by_model(model)[-1][:60]}  payments: {PAYMENTS}")

> **Observe:** the first four rows match the documented fail-closed rule. The last row is an **observed** detail of `openai-agents` 0.23.1. The human-in-the-loop docs page doesn't list it, but the `FunctionTool.needs_approval` docstring says callables receive arguments only when validation preserves their values and types. Here an integer `39` for a `float` parameter gets coerced, so the policy is skipped and the call pauses. Models often emit `39` rather than `39.0`. Result: small refunds that "should" auto-run end up in the review queue. It fails safe, but it inflates reviewer load.
>
> The approved NaN call still moved no money. Approval answers "may this call run?". It never makes an invalid amount valid, because code checks the amount before and after the human.
>
> **Try it:** define the tool with `amount_cents: int` (integer cents is also the safer money type) and re-run the last row with `{"order_id": "A-1001", "amount_cents": 3900}`. The policy is consulted again.

**Sticky and partial decisions.** `always_approve=True` (or `always_reject=True`) applies to later calls of the same tool by the same agent for the rest of this run, and survives serialization. When one turn produces several approval items, you can decide some and leave others: the resolved calls run, and the run pauses again on the rest.

In [ ]:
CANCELLED: list[str] = []


@function_tool(needs_approval=True)
def cancel_order(order_id: str) -> str:
    """Cancel an order that has not shipped."""
    CANCELLED.append(order_id)
    return json.dumps({"status": "cancelled", "order_id": order_id})


# Sticky: approve the first call with always_approve=True, round-trip through a string, and the second call runs unasked
model = ScriptedModel([[function_call("cancel_order", {"order_id": "A-1"}, call_id="c1")],
                       [function_call("cancel_order", {"order_id": "A-2"}, call_id="c2")],
                       [assistant_message("Both cancelled.")]])
agent = Agent(name="Ops", model=model, tools=[cancel_order])
result = await Runner.run(agent, "cancel A-1 and A-2")
state = result.to_state()
state.approve(result.interruptions[0], always_approve=True)
state = await RunState.from_string(agent, state.to_string())
result = await Runner.run(agent, state)
print("sticky:  ", result.final_output, "| interruptions:", result.interruptions, "| cancelled:", CANCELLED)

# Partial: two calls in one turn, decide only one of them
CANCELLED.clear()
model = ScriptedModel([[function_call("cancel_order", {"order_id": "B-1"}, call_id="c1"),
                        function_call("cancel_order", {"order_id": "B-2"}, call_id="c2")],
                       [assistant_message("B-1 cancelled; B-2 was declined.")]])
agent = Agent(name="Ops", model=model, tools=[cancel_order])
result = await Runner.run(agent, "cancel B-1 and B-2")
state = result.to_state()
state.approve(result.interruptions[0])                       # decide c1 only
result = await Runner.run(agent, state)
print("partial: cancelled:", CANCELLED, "| still pending:", [i.call_id for i in result.interruptions])
state = result.to_state()
state.reject(result.interruptions[0], rejection_message="Customer changed their mind on B-2.")
result = await Runner.run(agent, state)
print("         final:", result.final_output, "| cancelled:", CANCELLED)

**Guardrail before or after the human?** By default, tool input guardrails run **after** approval. A reviewer can then be asked to approve a $900 refund that the gate rejects anyway. `ToolExecutionConfig(pre_approval_tool_input_guardrails=True)` runs the gate first.

In [ ]:
RUN_CONFIG = RunConfig(tool_execution=ToolExecutionConfig(pre_approval_tool_input_guardrails=True))

for label, config in [("default order", None), ("gate before approval", RUN_CONFIG)]:
    reset()
    model = ScriptedModel([[function_call("issue_refund", {"order_id": "A-2002", "amount_usd": 900.0}, call_id="r1")],
                           [assistant_message("That amount needs a specialist; I'm escalating.")]])
    agent = refund_agent(model)
    result = await Runner.run(agent, "Refund $900 on A-2002.", context=SupportCtx("C-42"), run_config=config)
    asked = bool(result.interruptions)
    if asked:   # the reviewer approves... and then the gate rejects anyway
        state = result.to_state()
        state.approve(result.interruptions[0])
        result = await Runner.run(agent, state, run_config=config)
    print(f"{label:<21} reviewer asked: {asked!s:<5} model saw: {seen_by_model(model)[-1][:60]}  payments: {PAYMENTS}")

> **Observe:** the money never moves in either case, but with the default order a human spent time approving a request that policy forbids. Set the order explicitly in every approval design. Also remember the scope: `needs_approval` lives on function tools (and `Agent.as_tool`, `ShellTool`, `ApplyPatchTool`, `CustomTool`). Approvals for nested agents surface on the outer result; with `Runner.run_streamed`, drain `stream_events()` before reading `interruptions`; Realtime sessions use `approve_tool_call` / `reject_tool_call`.

## 9.5 Pause, serialize, resume: long-running reviews

Reviews take hours or days. `result.to_state().to_string()` (or `to_json()`) gives you a durable snapshot; `await RunState.from_string(agent, blob, context_override=...)` restores it in another process. The snapshot holds pending calls, arguments, decisions **and your context**, and it is not authenticated: keep it on the server. [Theory](README.md#95-pause-serialize-resume-long-running-reviews)

In [ ]:
@dataclass
class LeakyCtx:
    customer_id: str
    crm_api_token: str        # a secret that should never be in the context


reset()
model = ScriptedModel([
    [function_call("issue_refund", {"order_id": "A-2002", "amount_usd": 300.0}, call_id="r1")],
    [assistant_message("Your $300 refund is on its way.")],
])
agent = refund_agent(model)
paused = await Runner.run(agent, "Refund $300 on A-2002.", context=LeakyCtx("C-42", "sk-crm-SECRET"), run_config=RUN_CONFIG)

blob = paused.to_state().to_string()
snapshot = json.loads(blob)
print("schema version:", snapshot["$schemaVersion"], "| size:", len(blob), "chars")
print("top-level keys:", list(snapshot)[:12], "...")
print("pending call in blob:", '"r1"' in blob, "| arguments in blob:", "A-2002" in blob)
print("secret in blob:", "sk-crm-SECRET" in blob)

safe_blob = paused.to_state().to_string(context_serializer=lambda c: {"customer_id": c.customer_id})
print("secret with context_serializer:", "sk-crm-SECRET" in safe_blob)

In [ ]:
# Restore WITHOUT rebuilding the context: a dataclass comes back as a plain dict (the SDK logs a warning)
restored = await RunState.from_string(agent, safe_blob)
print("context type without override:", type(restored._context.context).__name__)   # peek at a private attr, demo only

# Restore WITH context_override, rebuilt from trusted server data (not from the blob)
restored = await RunState.from_string(agent, safe_blob, context_override=SupportCtx(customer_id="C-42"))
print("context type with override:   ", type(restored._context.context).__name__)
print("pending:", [(i.call_id, i.name, i.arguments) for i in restored.get_interruptions()])

Why a review must be **consumed exactly once**: the same stored snapshot, resumed twice (a double click, a retried request), executes the tool twice. A goodwill-voucher tool has no natural duplicate check, so the second resume really issues a second voucher.

In [ ]:
VOUCHERS: list[tuple[str, float]] = []


@function_tool(needs_approval=True)
def issue_voucher(customer_id: str, amount_usd: float) -> str:
    """Issue a goodwill voucher to a customer."""
    VOUCHERS.append((customer_id, amount_usd))
    return json.dumps({"voucher": f"V-{len(VOUCHERS)}", "amount_usd": amount_usd})


model = ScriptedModel([[function_call("issue_voucher", {"customer_id": "C-42", "amount_usd": 25.0}, call_id="v1")],
                       [assistant_message("Voucher sent.")], [assistant_message("Voucher sent.")]])
voucher_agent = Agent(name="Goodwill", model=model, tools=[issue_voucher])
voucher_blob = (await Runner.run(voucher_agent, "Sorry for the delay, send a $25 voucher")).to_state().to_string()

for attempt in (1, 2):   # the same stored blob, resumed twice
    state = await RunState.from_string(voucher_agent, voucher_blob)
    state.approve(state.get_interruptions()[0])
    await Runner.run(voucher_agent, state)
    print(f"resume #{attempt}: vouchers = {VOUCHERS}")

> **Observe:** two vouchers from one approval. The refund tool would have survived this because `_refund` refuses an already-refunded order, but you can't rely on every tool having such a check (in production, also give the payment call an idempotency key). The real fix belongs in your review store: an atomic `pending → resuming` transition, which the mini-project implements.

Store a version marker and an expiry with every snapshot and check both before resuming. Missing decisions and expired reviews **fail closed**.

In [ ]:
CURRENT_AGENT_VERSION = "refund-desk@3"


def resume_plan(record: dict, now: datetime) -> str:
    if record["status"] != "pending":
        return "refuse: already consumed"
    if now > record["expires_at"]:
        return "fail closed: expire the review, reject or escalate"
    if record["agent_version"] != CURRENT_AGENT_VERSION:
        return "route to the worker that runs " + record["agent_version"] + " (or drain before deploying)"
    return "resume"


t0 = datetime(2026, 10, 3, 9, 0, tzinfo=timezone.utc)
base = {"status": "pending", "agent_version": CURRENT_AGENT_VERSION, "expires_at": t0 + timedelta(hours=24)}
for label, record, now in [("fresh", base, t0 + timedelta(hours=2)),
                           ("expired", base, t0 + timedelta(hours=30)),
                           ("old code", {**base, "agent_version": "refund-desk@2"}, t0),
                           ("replayed", {**base, "status": "done"}, t0)]:
    print(f"{label:<9} -> {resume_plan(record, now)}")

**Without the SDK.** In a raw Responses API loop, "pausing" is just not executing a `function_call` item. Store the response ID (or your conversation) plus the `call_id`; when the decision arrives, send a `function_call_output` for that `call_id`. The same shape works for approve (real result) and reject (an error the model can act on).

In [ ]:
INPUT_ITEM = TypeAdapter(ResponseInputItemParam)

paused_response = Response.model_validate({
    "id": "resp_mock_1", "object": "response", "created_at": 0, "model": MODEL,
    "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
    "output": [{"type": "function_call", "id": "fc_1", "call_id": "call_77", "name": "issue_refund",
                "arguments": "{\"order_id\": \"A-2002\", \"amount_usd\": 300.0}", "status": "completed"}],
})
pending_call = next(i for i in paused_response.output if i.type == "function_call")
stored = {"response_id": paused_response.id, "call_id": pending_call.call_id, "name": pending_call.name,
          "arguments": pending_call.arguments}            # your review row; nothing executed yet
print("stored for review:", stored)

rejection = {"type": "function_call_output", "call_id": stored["call_id"],
             "output": json.dumps({"error": "rejected_by_reviewer", "message": "A supervisor declined this refund."})}
INPUT_ITEM.validate_python(rejection)                     # shape check against the SDK types
followup_kwargs = {"model": MODEL, "previous_response_id": stored["response_id"], "input": [rejection]}
print("follow-up request:", json.dumps(followup_kwargs, indent=2))

> **Note:** with `previous_response_id`, resuming depends on stored responses (30-day default retention, Chapter 1). For queues that can wait longer, keep local history instead. The managed **Agents API** (beta: `client.beta.agents`, `OpenAI-Beta: agents=v1` header) has its own pause: a session emits `agent.session.requires_action` with `required_actions`, and you answer with the same `turn_id` and `call_id`. After a restart, retrieve the session to find pending actions. How long a call may stay pending there is not documented; treat multi-day pauses as unverified and keep your own fail-closed expiry.

## 9.6 MCP approvals: `require_approval` and `mcp_approval_request`

Each MCP surface has its own approval loop. Responses API: the remote `mcp` tool requests approval **by default** and emits `mcp_approval_request` items that you answer with `mcp_approval_response`. Agents SDK: `HostedMCPTool` surfaces approvals as interruptions or calls `on_approval_request`; local MCP servers take `require_approval=`. [Theory](README.md#96-mcp-approvals-require_approval-and-mcp_approval_request)

In [ ]:
TOOL_PARAM = TypeAdapter(ToolParam)
READ_ONLY = ["search_contacts", "get_contact"]

crm_mcp_tool = {
    "type": "mcp", "server_label": "crm", "server_url": "https://crm.example.com/mcp",
    "allowed_tools": READ_ONLY + ["update_contact", "delete_contact"],
    "require_approval": {"never": {"tool_names": READ_ONLY}},    # every other tool still asks
    "authorization": "<oauth token from your vault>",          # resend on EVERY request; never stored by the API
}
TOOL_PARAM.validate_python(crm_mcp_tool)
print("split-policy mcp tool validates")

mcp_response = Response.model_validate({
    "id": "resp_mock_2", "object": "response", "created_at": 0, "model": MODEL,
    "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
    "output": [{"type": "mcp_approval_request", "id": "mcpr_9", "server_label": "crm", "name": "update_contact",
                "arguments": "{\"contact_id\": \"7\", \"email\": \"new@example.com\"}"}],
})


def review_mcp_request(req) -> tuple[bool, str]:
    """Stand-in for a human decision UI. Escape model-generated fields before showing them."""
    print(f"  REVIEW  server={html.escape(req.server_label)} tool={html.escape(req.name)} args={html.escape(req.arguments)}")
    return True, "Supervisor approved the e-mail change."


approvals = []
for req in (i for i in mcp_response.output if i.type == "mcp_approval_request"):
    approve, reason = review_mcp_request(req)
    approvals.append({"type": "mcp_approval_response", "approval_request_id": req.id, "approve": approve, "reason": reason})
for a in approvals:
    INPUT_ITEM.validate_python(a)

followup = {"model": MODEL, "tools": [crm_mcp_tool], "previous_response_id": mcp_response.id, "input": approvals}
print("follow-up input:", json.dumps(followup["input"], indent=2))
# client.responses.create(**followup) would continue the same conversation; the API then calls the MCP server.

In the Agents SDK, a `HostedMCPTool` produces the same approval request. Without a callback, it becomes an ordinary interruption you approve or reject on the `RunState`; with `on_approval_request`, your function decides immediately and the run never pauses. The scripted model below emits a real `McpApprovalRequest` item, exactly what the API would return.

In [ ]:
approval_request = McpApprovalRequest(id="mcpr_9", type="mcp_approval_request", server_label="crm",
                                      name="update_contact", arguments='{"contact_id": "7", "email": "new@example.com"}')
hosted_config = {"type": "mcp", "server_label": "crm", "server_url": "https://crm.example.com/mcp", "require_approval": "always"}

# 1) Manual: the run pauses, a human decides
model = ScriptedModel([[approval_request], [assistant_message("E-mail updated.")]])
agent = Agent(name="CRM agent", model=model, tools=[HostedMCPTool(tool_config=hosted_config)])
result = await Runner.run(agent, "Change contact 7's e-mail to new@example.com")
print("manual:   interruptions:", [(i.name, i.call_id) for i in result.interruptions])
state = result.to_state()
state.approve(result.interruptions[0])
result = await Runner.run(agent, state)
sent = [x for x in model.last_call.input if isinstance(x, dict) and x.get("type") == "mcp_approval_response"]
print("          sent to the API:", sent, "| final:", result.final_output)


# 2) Callback: an automated decision, no pause, no human
def auto_policy(request) -> dict:
    name = request.data.name
    return {"approve": name in READ_ONLY, "reason": "Writes need a supervisor." if name not in READ_ONLY else "read-only"}


model = ScriptedModel([[approval_request], [assistant_message("I can't change e-mails; a supervisor must do it.")]])
agent = Agent(name="CRM agent", model=model, tools=[HostedMCPTool(tool_config=hosted_config, on_approval_request=auto_policy)])
result = await Runner.run(agent, "Change contact 7's e-mail")
sent = [x for x in model.last_call.input if isinstance(x, dict) and x.get("type") == "mcp_approval_response"]
print("callback: interruptions:", result.interruptions, "| sent:", sent)

# 3) A local MCP server (you connect): require_approval on the server object. Constructing it does not connect.
local_crm = MCPServerStreamableHttp(params={"url": "https://crm.internal.example/mcp"}, name="crm-internal",
                                    require_approval={"never": {"tool_names": READ_ONLY},
                                                      "always": {"tool_names": ["update_contact", "delete_contact"]}})
print("local server configured:", local_crm.name)

> **Observe:** the callback path is an **automated** decision. It is fine for low-risk calls; a callback that approves everything has the latency of approvals and none of the protection. Sticky hosted-MCP decisions are keyed by `server_label` **and** tool name.

## 9.7 Structured handoff to a human

The human sees the handoff, not the transcript. Make it a strict schema (required + nullable fields, an enum for reasons), let the model write what only it can write (summary, request, recommendation), and fill the facts from the run and from trusted data in code. [Theory](README.md#97-structured-handoff-to-a-human)

Here the escalate tool only records the model's **draft**. The application finalizes it after the run, with the run's `new_items` in hand, and only then opens the ticket.

In [ ]:
@function_tool(name_override="escalate_to_human")
def escalate_draft(handoff: HumanHandoff) -> str:
    """Hand the case to a human agent with a complete, self-contained summary. Use when the customer asks for
    a person, the request is not covered by policy, a tool says a human is needed, or you cannot make progress."""
    return json.dumps({"status": "handoff_received"})   # the app finalizes and files the ticket


def draft_from_run(new_items) -> dict | None:
    for item in new_items:
        if item.type == "tool_call_item" and _name(item.raw_item) == "escalate_to_human":
            return json.loads(item.raw_item.arguments)["handoff"]
    return None


def finalize_handoff(draft: dict, ctx: SupportCtx, new_items, prior_actions: list[str] = ()) -> HumanHandoff:
    """prior_actions: your server-side ledger of earlier user turns (new_items holds only this run)."""
    data = dict(draft)
    data["customer_id"] = ctx.customer_id                                        # from auth, overrides the model
    data["actions_taken"] = list(prior_actions) + actions_from_run(new_items)    # from the runs, not from recollection
    if data.get("order_id") and owned_order(ctx, data["order_id"]) is None:
        data["order_id"] = None                  # hallucinated, or another customer's order: never pass it on
    return HumanHandoff.model_validate(data)


reset()
model_draft = {  # what the model wrote: wrong customer, a made-up order ID, and an action that never happened
    "customer_id": "C-99", "order_id": "A-2O02", "escalation_reason": "policy_limit",
    "issue_summary": "Customer received a damaged espresso machine ($640) and wants a full refund. Above the $500 agent limit.",
    "customer_request": "Full refund, no replacement",
    "actions_taken": ["Issued a $640 refund"],
    "amount_usd": 640.0, "recommended_action": "Approve full refund; photos of damage provided in chat",
    "customer_sentiment_note": "Frustrated, polite"}
model = ScriptedModel([
    [function_call("lookup_order", {"order_id": "A-2002"}, call_id="look_1")],
    [function_call("issue_refund", {"order_id": "A-2002", "amount_usd": 640.0}, call_id="r1")],
    [function_call("escalate_to_human", {"handoff": model_draft}, call_id="esc_1")],
])
agent = Agent[SupportCtx](name="Refund desk", model=model, tools=[lookup_order, issue_refund, escalate_draft],
                          tool_use_behavior=StopAtTools(stop_at_tool_names=["escalate_to_human"]))
ctx = SupportCtx("C-42")
result = await Runner.run(agent, "My espresso machine A-2002 arrived broken. Refund all $640.", context=ctx, run_config=RUN_CONFIG)

draft = draft_from_run(result.new_items)
final = finalize_handoff(draft, ctx, result.new_items)
ticket_id = create_ticket(final.model_dump())
print("model draft actions:", draft["actions_taken"], "| customer:", draft["customer_id"], "| order:", draft["order_id"])
print(f"\nTicket {ticket_id}:")
print(final.model_dump_json(indent=2))
print("payments:", PAYMENTS)

> **Observe:** the model claimed it "Issued a $640 refund". The run log shows the gate rejected that call, and `PAYMENTS` is empty. Code replaced the claim with the facts, fixed the customer ID from the authenticated context, and nulled the order ID that does not exist (`A-2O02` with a letter O). Another customer's order ID (`B-7007`) would be nulled the same way, because the check is ownership, not existence. The real order ID still sits in `actions_taken` for the human. In a multi-turn conversation, pass the earlier turns' facts as `prior_actions` from your server-side ledger (9.2), because `new_items` holds only this run.
>
> **Try it:** extend `finalize_handoff` to fill `amount_usd` from the last `issue_refund` call's arguments when the model left it `null`.

## 9.8 Confidence calibration with labeled sets

A confidence score is useful only if it is **calibrated**: among items scored 0.9, about 90% should be correct. You learn that from a labeled validation set, per field, and you pick the **lowest** threshold that still meets a target precision with enough support. [Theory](README.md#98-confidence-calibration-with-labeled-sets)

The labeled set below is synthetic: an extraction pipeline outputs a model-reported confidence per field, and a reviewer marked each extraction right or wrong. Amount scores are conservative, date scores are roughly calibrated, and IBAN scores are overconfident with an error floor that no score reflects.

In [ ]:
def make_labeled_set(n_per_field: int = 1500, seed: int = 11) -> list[dict]:
    rng = random.Random(seed)
    truth = {   # P(correct | score): the hidden behavior that calibration has to discover
        "amount": lambda s: 1 - (1 - s) * 0.5,             # conservative: more accurate than the score says
        "invoice_date": lambda s: s,                      # roughly calibrated
        "iban": lambda s: min(0.85, 1 - (1 - s) * 2),     # overconfident, with an OCR error floor
    }
    rows = []
    for fld, p_correct in truth.items():
        for _ in range(n_per_field):
            score = round(min(0.999, 0.45 + 0.55 * rng.betavariate(5, 1.4)), 3)
            rows.append({"field": fld, "confidence": score, "correct": rng.random() < p_correct(score)})
    rng.shuffle(rows)
    return rows


def pick_threshold(scored: list[tuple[float, bool]], target_precision: float = 0.98,
                   min_support: int = 50) -> float | None:
    """Lowest threshold whose auto-approved set meets the target precision (one field)."""
    for t in sorted({s for s, _ in scored}):
        above = [ok for s, ok in scored if s >= t]
        if len(above) < min_support:
            break
        if sum(above) / len(above) >= target_precision:
            return t
    return None


def reliability(scored, bins=(0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.01)) -> list[tuple[str, int, float]]:
    rows = []
    for lo, hi in zip(bins, bins[1:]):
        bucket = [ok for s, ok in scored if lo <= s < hi]
        if bucket:
            rows.append((f"[{lo:.2f}, {hi:.2f})", len(bucket), sum(bucket) / len(bucket)))
    return rows


labeled = make_labeled_set()
calib, holdout = labeled[: len(labeled) // 2], labeled[len(labeled) // 2:]
for fld in ("amount", "invoice_date", "iban"):
    print(f"\n{fld}: reliability on the calibration split (bin, n, accuracy)")
    scored = [(r["confidence"], r["correct"]) for r in calib if r["field"] == fld]
    for b, n, acc in reliability(scored):
        print(f"   {b:<14} n={n:<4} acc={acc:.2f}")

In [ ]:
TARGET = 0.95
print(f"target precision {TARGET:.0%} on auto-approved items, min support 50\n")
print(f"{'field':<13} {'threshold':>9} {'holdout precision':>18} {'to human review':>16} {'naive >=0.8 precision':>22}")
for fld in ("amount", "invoice_date", "iban"):
    cal = [(r["confidence"], r["correct"]) for r in calib if r["field"] == fld]
    test = [(r["confidence"], r["correct"]) for r in holdout if r["field"] == fld]
    t = pick_threshold(cal, TARGET)
    naive = [ok for s, ok in test if s >= 0.8]
    if t is None:
        print(f"{fld:<13} {'none':>9} {'-':>18} {'100%':>16} {sum(naive) / len(naive):>22.2f}")
        continue
    above = [ok for s, ok in test if s >= t]
    review_share = 1 - len(above) / len(test)
    print(f"{fld:<13} {t:>9.3f} {sum(above) / len(above):>18.3f} {review_share:>16.0%} {sum(naive) / len(naive):>22.2f}")

> **Observe:** each field gets its own threshold, and IBAN gets **none**: no cut-off reaches the target, so every IBAN goes to review until the extractor improves. The date threshold reaches the target on the calibration split but lands just under it on the held-out split: the margin is thin, so raise it a notch or label more data before going live. The "naive ≥ 0.8" column is what a "self-rated 8 of 10 or better" rule would deliver. The review share is your human workload, so put it in the solution plan next to the target precision. Re-check on the held-out split, and re-calibrate when the model, prompt or document mix changes.

**Token log probabilities** are another signal, available on the Responses API only when reasoning effort is `none` (GPT-6.1 Terra, GPT-6 Sol and GPT-6 Luna support it; GPT-6 Astra and GPT-6.1 Sol don't, and the GPT-6 migration guide says to remove logprobs at any other effort). Below: the request shape, and a confidence computed from a mock response. Like any score, it must be calibrated before it gates anything.

In [ ]:
import inspect

logprob_request = {
    "model": MODEL,                                   # gpt-6-luna supports effort "none"
    "reasoning": {"effort": "none"},
    "include": ["message.output_text.logprobs"],
    "top_logprobs": 3,
    "input": "Extract the invoice date as YYYY-MM-DD: 'Invoice dated 28 Sept 2026'",
}
create_params = inspect.signature(OpenAI(api_key="unused").responses.create).parameters
print("all request keys exist in the SDK:", all(k in create_params for k in logprob_request))

mock = Response.model_validate({
    "id": "resp_mock_3", "object": "response", "created_at": 0, "model": MODEL,
    "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
    "output": [{"type": "message", "id": "msg_1", "role": "assistant", "status": "completed", "content": [{
        "type": "output_text", "text": "2026-09-28", "annotations": [],
        "logprobs": [{"token": t, "logprob": lp, "bytes": [], "top_logprobs": []}
                     for t, lp in [("2026", -0.001), ("-", -0.0001), ("09", -0.02), ("-", -0.0001), ("28", -0.41)]]}]}],
})
lps = [lp.logprob for lp in mock.output[0].content[0].logprobs]
print(f"sequence probability {math.exp(sum(lps)):.2f}, weakest token probability {min(math.exp(x) for x in lps):.2f}")

## 9.9 Stratified sampling for human review

A calibrated threshold sends uncertain items to people. You still need to audit the **auto-approved** items, and an aggregate accuracy hides weak segments. Stratify (document type × field × confidence band), sample every stratum with a minimum, and report accuracy per stratum. [Theory](README.md#99-stratified-sampling-for-human-review)

In [ ]:
def make_auto_approved(n: int = 20000, seed: int = 5) -> list[dict]:
    rng = random.Random(seed)
    doc_mix = [("invoice", 0.70, 0.985), ("receipt", 0.26, 0.975), ("handwritten_note", 0.04, 0.60)]
    items = []
    for i in range(n):
        r, acc_for = rng.random(), None
        cum = 0.0
        for doc, share, acc in doc_mix:
            cum += share
            if r < cum:
                doc_type, acc_for = doc, acc
                break
        else:
            doc_type, acc_for = doc_mix[-1][0], doc_mix[-1][2]
        items.append({"id": i, "doc_type": doc_type, "field": rng.choice(["amount", "date"]),
                      "confidence_band": rng.choice(["0.90-0.95", "0.95-1.00"]), "correct": rng.random() < acc_for})
    return items


def stratified_sample(items: list[dict], strata_keys: tuple[str, ...], rate: float = 0.02,
                      min_per_stratum: int = 20, seed: int = 7) -> list[dict]:
    """Audit sample: proportional rate, but at least min_per_stratum from each stratum."""
    rng = random.Random(seed)
    groups: dict[tuple, list[dict]] = defaultdict(list)
    for item in items:
        groups[tuple(item[k] for k in strata_keys)].append(item)
    sample = []
    for _, group in sorted(groups.items()):
        n = min(len(group), max(min_per_stratum, round(rate * len(group))))
        sample.extend(rng.sample(group, n))
    return sample


auto_approved = make_auto_approved()
overall = sum(i["correct"] for i in auto_approved) / len(auto_approved)
print(f"auto-approved items: {len(auto_approved)}, true overall accuracy {overall:.1%} (unknown in production)")

uniform = random.Random(7).sample(auto_approved, round(0.01 * len(auto_approved)))
hw = [i for i in uniform if i["doc_type"] == "handwritten_note"]
print(f"uniform 1% sample: {len(uniform)} items, accuracy {sum(i['correct'] for i in uniform) / len(uniform):.1%}, "
      f"handwritten notes in it: {len(hw)}")

KEYS = ("doc_type", "field", "confidence_band")
audit = stratified_sample(auto_approved, KEYS, rate=0.01, min_per_stratum=20)
print(f"stratified sample: {len(audit)} items\n")
by_stratum: dict[tuple, list[bool]] = defaultdict(list)
for i in audit:
    by_stratum[tuple(i[k] for k in KEYS)].append(i["correct"])
print(f"{'stratum':<44} {'n':>4} {'accuracy':>9}")
for stratum, oks in sorted(by_stratum.items()):
    acc = sum(oks) / len(oks)
    print(f"{' / '.join(stratum):<44} {len(oks):>4} {acc:>9.0%} {'<-- below 95%: route to review' if acc < 0.95 else ''}")

> **Observe:** the aggregate is about 97%, and the uniform sample contains only a handful of handwritten notes, too few to notice anything. The stratified sample guarantees 20 per stratum and every handwritten stratum lights up. Every corrected item in the audit is a new labeled case for the eval set (Chapter 6) and the next calibration run (9.8).
>
> **Try it:** to estimate overall accuracy from the stratified sample, weight each stratum's accuracy by its share of the population. Compare with the unweighted sample mean.

## 9.10 Operating human-in-the-loop in production

A HITL design needs operations: an audit trail of every request and decision, metrics that tell you when thresholds are wrong, graduated autonomy, timeouts that fail closed, platform stops, and separation of duties. [Theory](README.md#910-operating-human-in-the-loop-in-production)

In [ ]:
def make_audit_log(seed: int = 3) -> list[dict]:
    rng = random.Random(seed)
    t0 = datetime(2026, 9, 1, tzinfo=timezone.utc)
    log = []
    for i in range(400):
        tool = rng.choices(["issue_refund", "cancel_order", "update_address"], [0.5, 0.3, 0.2])[0]
        expired = rng.random() < 0.03
        decision = None if expired else ("approve" if rng.random() < {"issue_refund": 0.86, "cancel_order": 0.995,
                                                                         "update_address": 0.6}[tool] else "reject")
        log.append({"review_id": f"rv_{i}", "tool": tool, "requested_at": t0 + timedelta(minutes=37 * i),
                    "decided_at": None if expired else t0 + timedelta(minutes=37 * i + rng.randint(2, 600)),
                    "decision": decision, "reviewer": None if expired else rng.choice(["sup_ana", "sup_ben", "sup_cho"])})
    return log


def hitl_metrics(log: list[dict]) -> None:
    by_tool: dict[str, list[dict]] = defaultdict(list)
    for r in log:
        by_tool[r["tool"]].append(r)
    print(f"{'tool':<15} {'requests':>8} {'approve%':>9} {'median min':>11} {'expired':>8}  signal")
    for tool, rows in sorted(by_tool.items()):
        decided = [r for r in rows if r["decision"]]
        approve = sum(r["decision"] == "approve" for r in decided) / len(decided)
        waits = sorted((r["decided_at"] - r["requested_at"]).total_seconds() / 60 for r in decided)
        expired = sum(r["decision"] is None for r in rows)
        signal = ("threshold too low? reviewers rubber-stamp" if approve > 0.99 else
                  "prompt or policy-gate problem?" if approve < 0.7 else "")
        print(f"{tool:<15} {len(rows):>8} {approve:>9.1%} {waits[len(waits) // 2]:>11.0f} {expired:>8}  {signal}")


hitl_metrics(make_audit_log())

Two more operational rules in code: **separation of duties** (the reviewer must not be the beneficiary), and **platform stops**: for GPT-6 Astra agent workflows, a `403` with `misalignment_policy_violation` means stop dispatching actions for that conversation, don't auto-retry, and route it to a human. The misalignment monitoring guide documents the HTTP `403`, the error type `invalid_request_error` and the code `misalignment_policy_violation`, and says to match the code rather than the message text. The message text in the fake error below is made up.

In [ ]:
def may_decide(reviewer: dict, review: dict) -> bool:
    return reviewer["role"] == "supervisor" and reviewer.get("customer_account") != review["customer_id"]


print("supervisor, unrelated:", may_decide({"role": "supervisor", "customer_account": None}, {"customer_id": "C-42"}))
print("supervisor = customer:", may_decide({"role": "supervisor", "customer_account": "C-42"}, {"customer_id": "C-42"}))


def route_platform_error(exc: Exception) -> str:
    if isinstance(exc, openai.PermissionDeniedError) and exc.code == "misalignment_policy_violation":   # match the code, not the text
        return "stop actions for this conversation; no auto-retry; escalate to a human"
    if isinstance(exc, (openai.RateLimitError, openai.APITimeoutError)):
        return "retry with backoff"
    return "raise"


fake_403 = openai.PermissionDeniedError(
    "misalignment_policy_violation: request blocked",
    response=httpx.Response(403, request=httpx.Request("POST", "https://api.openai.com/v1/responses")),
    body={"type": "invalid_request_error", "code": "misalignment_policy_violation", "message": "request blocked"},
)
print("403 ->", route_platform_error(fake_403))

> **Note: avoid / migrate.** Approval flows built in **Agent Builder** (deprecated, shuts down 2026-11-30) move to the Agents SDK patterns in this notebook or to ChatGPT Workspace Agents. Assistants API run-step flows are retired; don't port them.

## Mini-project — Approval-gated refund desk

Everything together:

- An agent with `lookup_order` (ownership-checked), `issue_refund` (`needs_approval` above $100) and `escalate_to_human` (strict `HumanHandoff`, run stops on it, and no side effect may run in the same response).
- A blocking input guardrail for explicit human requests and the deterministic policy gate (ownership, valid positive amount, duplicates, $500 hard limit), run **before** approval.
- A turn limit whose `max_turns` error handler opens a `cannot_progress` ticket, on the first run and on every resume.
- A server-side **review store**: it pauses and serializes the run, "restarts the process" (a brand-new agent object on resume), authenticates and authorizes the reviewer, accepts decisions only for server-owned pending calls, refuses snapshots from a different agent version, consumes each review once, and fails closed on expiry.
- Escalations become tickets with a handoff finalized from the run.

In [ ]:
DESK_VERSION = "refund-desk@1"
DESK_MAX_TURNS = 6
DESK_INSTRUCTIONS = ESCALATION_INSTRUCTIONS + """
# Refunds
Look up the order first. Refund at most the order total. Refunds above $100 need a supervisor's approval;
the system asks for it, you just call issue_refund. If a tool says a human is needed, escalate."""


def build_desk(model) -> Agent[SupportCtx]:
    """Called on every request and every resume: nothing in the agent object survives a restart."""
    return Agent[SupportCtx](
        name="Refund desk", instructions=DESK_INSTRUCTIONS, model=model,
        tools=[lookup_order, issue_refund, escalate_draft],
        input_guardrails=[explicit_human_request],
        hooks=EscalationWins(),                                    # no refund in the same response as a handoff
        model_settings=ModelSettings(parallel_tool_calls=False),
        tool_use_behavior=StopAtTools(stop_at_tool_names=["escalate_to_human"]),
    )


def escalate_desk_on_max_turns(data: RunErrorHandlerInput) -> RunErrorHandlerResult:
    """Failure threshold: the desk ran out of turns. Open a cannot_progress ticket from the run's facts."""
    ctx: SupportCtx = data.context.context
    handoff = HumanHandoff(
        customer_id=ctx.customer_id, order_id=None, escalation_reason="cannot_progress",
        issue_summary=f"The refund desk hit its {DESK_MAX_TURNS}-turn limit without resolving the request.",
        customer_request="See actions taken", actions_taken=actions_from_run(data.run_data.new_items),
        amount_usd=None, recommended_action="Review the actions taken and contact the customer",
        customer_sentiment_note=None)
    outcome = {"status": "escalated", "ticket": create_ticket(handoff.model_dump()), "reason": "cannot_progress"}
    return RunErrorHandlerResult(final_output=outcome, include_in_history=False)


DESK_LIMITS = {"max_turns": DESK_MAX_TURNS, "error_handlers": {"max_turns": escalate_desk_on_max_turns}}


@dataclass
class Review:
    review_id: str
    state: str                  # the RunState snapshot: stays on the server
    customer_id: str
    agent_version: str
    pending: list[dict]         # server-owned copy of what the reviewer is asked about
    expires_at: datetime
    status: str = "pending"     # pending -> resuming -> done | expired
    decided_by: str | None = None


class ReviewStore:
    def __init__(self) -> None:
        self._rows: dict[str, Review] = {}

    def create(self, state: str, customer_id: str, pending: list[dict], now: datetime) -> str:
        review_id = uuid.uuid4().hex                       # opaque ID: the only thing the browser gets
        self._rows[review_id] = Review(review_id, state, customer_id, DESK_VERSION, pending, now + timedelta(hours=24))
        return review_id

    def public_view(self, review_id: str) -> dict:
        r = self._rows[review_id]
        return {"review_id": r.review_id, "status": r.status, "expires_at": r.expires_at.isoformat(),
                "calls": [{"call_id": p["call_id"], "tool": html.escape(p["tool"]), "arguments": html.escape(p["arguments"])}
                          for p in r.pending]}

    def claim(self, review_id: str, reviewer: dict, call_ids: set[str], now: datetime, agent_version: str) -> Review:
        """Authorize, validate, and atomically move pending -> resuming (a DB transaction in production)."""
        r = self._rows.get(review_id)
        if r is None:
            raise KeyError("unknown review")
        if not may_decide(reviewer, {"customer_id": r.customer_id}):
            raise PermissionError("reviewer may not decide this review")
        if r.status != "pending":
            raise RuntimeError(f"review already {r.status}")
        if r.agent_version != agent_version:   # stays pending: route it to the worker running r.agent_version
            raise LookupError(f"review was created by {r.agent_version}; this worker runs {agent_version}")
        unknown = call_ids - {p["call_id"] for p in r.pending}
        if unknown:
            raise ValueError(f"decisions for calls that are not pending: {sorted(unknown)}")
        if now > r.expires_at:
            r.status = "expired"
            raise TimeoutError("review expired")
        r.status, r.decided_by = "resuming", reviewer["id"]
        return r

    def complete(self, review_id: str) -> None:
        self._rows[review_id].status = "done"


class RefundDesk:
    def __init__(self, model, store: ReviewStore) -> None:
        self.model, self.store = model, store

    async def handle(self, text: str, ctx: SupportCtx, now: datetime) -> dict:
        try:
            result = await Runner.run(build_desk(self.model), text, context=ctx, run_config=RUN_CONFIG, **DESK_LIMITS)
        except InputGuardrailTripwireTriggered:
            handoff = HumanHandoff(customer_id=ctx.customer_id, order_id=None, escalation_reason="customer_requested_human",
                                   issue_summary="Customer asked for a person on their first message.",
                                   customer_request=text, actions_taken=[], amount_usd=None,
                                   recommended_action="Contact the customer", customer_sentiment_note=None)
            return {"status": "escalated", "ticket": create_ticket(handoff.model_dump())}
        return self._finish(result, ctx, now)

    async def decide(self, reviewer: dict, review_id: str, decisions: dict[str, bool], now: datetime) -> dict:
        review = self.store.claim(review_id, reviewer, set(decisions), now, agent_version=DESK_VERSION)
        agent = build_desk(self.model)                     # a fresh process would do exactly this
        state = await RunState.from_string(agent, review.state,
                                           context_override=SupportCtx(customer_id=review.customer_id))
        for item in state.get_interruptions():             # server-owned items only
            if decisions.get(item.call_id) is True:
                state.approve(item)
            else:                                          # missing decision -> reject (fail closed)
                state.reject(item, rejection_message="A supervisor declined this refund. Offer to escalate.")
        result = await Runner.run(agent, state, run_config=RUN_CONFIG, **DESK_LIMITS)
        self.store.complete(review_id)
        return self._finish(result, SupportCtx(review.customer_id), now)

    def _finish(self, result, ctx: SupportCtx, now: datetime) -> dict:
        if isinstance(result.final_output, dict) and result.final_output.get("reason") == "cannot_progress":
            return result.final_output                     # the max_turns handler already opened the ticket
        if result.interruptions:
            pending = [{"call_id": i.call_id, "tool": i.name, "arguments": i.arguments} for i in result.interruptions]
            blob = result.to_state().to_string(context_serializer=lambda c: {"customer_id": c.customer_id})
            review_id = self.store.create(blob, ctx.customer_id, pending, now)
            return {"status": "pending_review", "review": self.store.public_view(review_id)}
        draft = draft_from_run(result.new_items)
        if draft is not None:
            handoff = finalize_handoff(draft, ctx, result.new_items)
            return {"status": "escalated", "ticket": create_ticket(handoff.model_dump()), "reason": handoff.escalation_reason}
        return {"status": "answered", "reply": str(result.final_output)}

In [ ]:
NOW = datetime(2026, 10, 3, 9, 0, tzinfo=timezone.utc)
SUPERVISOR = {"id": "sup_ana", "role": "supervisor", "customer_account": None}
AGENT_USER = {"id": "agent_bo", "role": "agent", "customer_account": None}
ctx = SupportCtx("C-42")


def refund_call(order_id: str, amount: float, call_id: str):
    return [function_call("issue_refund", {"order_id": order_id, "amount_usd": amount}, call_id=call_id)]


# Scenario 1: $39, within the auto limit
reset()
desk = RefundDesk(ScriptedModel([refund_call("A-1001", 39.0, "r1"), [assistant_message("Refunded $39 for the phone case.")]]), ReviewStore())
print("1 small refund   ->", await desk.handle("Refund my phone case A-1001.", ctx, NOW), "| payments:", PAYMENTS)

# Scenario 2: $300 -> paused; decisions arrive later through an authenticated endpoint
reset()
store = ReviewStore()
desk = RefundDesk(ScriptedModel([refund_call("A-2002", 300.0, "r1"),
                                 [assistant_message("Your $300 refund was approved and is on its way.")]]), store)
paused = await desk.handle("Refund $300 on A-2002, it arrived damaged.", ctx, NOW)
print("2 large refund   ->", paused["status"], json.dumps(paused["review"]["calls"]))
rid = paused["review"]["review_id"]
for who, decisions, label in [(AGENT_USER, {"r1": True}, "non-supervisor"),
                              (SUPERVISOR, {"r1": True, "r99": True}, "unknown call ID")]:
    try:
        await desk.decide(who, rid, decisions, NOW + timedelta(hours=1))
    except (PermissionError, ValueError) as exc:
        print(f"  refused ({label}): {exc}")
print("  approved        ->", await desk.decide(SUPERVISOR, rid, {"r1": True}, NOW + timedelta(hours=3)), "| payments:", PAYMENTS)
try:
    await desk.decide(SUPERVISOR, rid, {"r1": True}, NOW + timedelta(hours=3))
except RuntimeError as exc:
    print("  refused (replay):", exc, "| payments:", PAYMENTS)

In [ ]:
# Scenario 3: $640 -> the gate rejects before any review; the model escalates with a draft that code finalizes
reset()
draft_640 = {**model_draft, "customer_id": "C-42", "order_id": "A-2002", "actions_taken": []}
desk = RefundDesk(ScriptedModel([refund_call("A-2002", 640.0, "r1"),
                                 [function_call("escalate_to_human", {"handoff": draft_640}, call_id="esc_1")]]), ReviewStore())
print("3 above limit    ->", await desk.handle("Refund all $640 on A-2002.", ctx, NOW))
print("  actions_taken  :", TICKETS[-1]["actions_taken"])

# Scenario 4: explicit request -> tripwire, no model call
reset()
desk = RefundDesk(ScriptedModel([]), ReviewStore())
print("4 asks for human ->", await desk.handle("I want to talk to a manager.", ctx, NOW))

# Scenario 5: the review sits too long -> fail closed
reset()
store = ReviewStore()
desk = RefundDesk(ScriptedModel([refund_call("A-2002", 300.0, "r1")]), store)
paused = await desk.handle("Refund $300 on A-2002.", ctx, NOW)
rid = paused["review"]["review_id"]
try:
    await desk.decide(SUPERVISOR, rid, {"r1": True}, NOW + timedelta(hours=30))
except TimeoutError as exc:
    print("5 expired review ->", exc, "| status:", store.public_view(rid)["status"], "| payments:", PAYMENTS)
    print("  next step: tell the customer, open a ticket with reason 'cannot_progress'")

In [ ]:
# Scenario 6: the model loops on lookups -> the turn limit becomes a cannot_progress ticket, not a crash
reset()
loop_script = [[function_call("lookup_order", {"order_id": "A-1001"}, call_id=f"l{i}")] for i in range(10)]
desk = RefundDesk(ScriptedModel(loop_script), ReviewStore())
print("6 agent loops    ->", await desk.handle("Where is my order?", ctx, NOW), "| tickets:", len(TICKETS))

# Scenario 7: a review created by an older deploy is not resumed with today's tools
reset()
store = ReviewStore()
desk = RefundDesk(ScriptedModel([refund_call("A-2002", 300.0, "r1"), [assistant_message("Refunded.")]]), store)
rid = (await desk.handle("Refund $300 on A-2002.", ctx, NOW))["review"]["review_id"]
store._rows[rid].agent_version = "refund-desk@0"          # simulate a snapshot written by the previous release
try:
    await desk.decide(SUPERVISOR, rid, {"r1": True}, NOW + timedelta(hours=1))
except LookupError as exc:
    print("7 old version    -> refused:", exc, "| status:", store.public_view(rid)["status"], "| payments:", PAYMENTS)

# Scenario 8: one model response escalates AND refunds -> the handoff wins, no money moves
reset()
mixed_draft = {**draft_640, "order_id": "A-1001", "escalation_reason": "customer_requested_human", "amount_usd": None,
               "issue_summary": "Customer asked for a refund on A-1001 or a person in the same message. Handed over before any refund.",
               "customer_request": "Refund A-1001, or talk to someone", "recommended_action": "Call back and decide on the A-1001 refund"}
both = [function_call("escalate_to_human", {"handoff": mixed_draft}, call_id="esc_1")] + refund_call("A-1001", 39.0, "r1")
desk = RefundDesk(ScriptedModel([both]), ReviewStore())
print("8 mixed response ->", await desk.handle("Refund A-1001, or just get me someone.", ctx, NOW), "| payments:", PAYMENTS)

> **Observe:** the money moved exactly once (scenario 2), and only after an authorized supervisor decided on a server-owned pending call. Wrong reviewers, invented call IDs, replays, expired reviews and snapshots from another agent version were all refused before the snapshot was even loaded. A looping model ends as a `cannot_progress` ticket, and a response that both escalates and refunds moves no money. The browser only ever saw `public_view`: an opaque ID and escaped call details.

In [ ]:
# (live) the same desk on a real model, ~ $0.01. Tracing goes to the Traces dashboard.
# Tip: models often emit "300" (an integer) for amount_usd; per 9.4 the SDK then skips the policy and asks a human.
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (see 00-prerequisites)"
set_tracing_disabled(False)
reset()
live_store = ReviewStore()
live_desk = RefundDesk(MODEL, live_store)
outcome = await live_desk.handle("Hi, my espresso machine (order A-2002) arrived damaged. Please refund $300.",
                                 SupportCtx("C-42"), datetime.now(timezone.utc))
print(json.dumps(outcome, indent=2, default=str))
if outcome["status"] == "pending_review":
    approved = await live_desk.decide(SUPERVISOR, outcome["review"]["review_id"], {c["call_id"]: True for c in outcome["review"]["calls"]},
                                      datetime.now(timezone.utc))
    print(json.dumps(approved, indent=2), "| payments:", PAYMENTS)

> **Try it:** (1) ask the live desk for $700 and check that it escalates with reason `policy_limit`; (2) write "I'm furious, this is the worst shop ever" without asking for a person and check that it does **not** escalate; (3) add an output guardrail that blocks "your refund is approved" unless a refund actually executed in this run.

## Recap

- Escalate on **observable** triggers: explicit request, policy gap, failure threshold, high-risk action, ambiguous identity. Not on sentiment or a self-rated 1–10. Rate tool risk in the plan.
- `escalate_to_human` is a strict tool; `StopAtTools` ends the run after it fires (no further model call), but sibling calls in the same response still run, so add `parallel_tool_calls=False` and a check in code; `error_handlers={"max_turns": ...}` turns "can't progress" into a ticket.
- Rules that must hold go into code: tool input guardrails (`reject_content`), blocking input guardrails, output guardrails, a check in the tool. Mind the first-agent boundary.
- `needs_approval` pauses; `interruptions` → `approve` / `reject(rejection_message=...)` → `Runner.run(agent, state)` resumes the same run. Callables fail closed; set `pre_approval_tool_input_guardrails=True`.
- `RunState` snapshots contain your context and are unauthenticated: server-side storage, `context_serializer` without secrets, `context_override` on load, version marker, expiry, consume once.
- MCP: `require_approval` split policies, `mcp_approval_request` → `mcp_approval_response`; `on_approval_request` is automation, not a human.
- Handoffs: the model drafts, code fills facts from `new_items` (this run), a server-side ledger (earlier turns) and trusted data, with ownership checks.
- Calibrate per field on labeled data; audit auto-approved output with stratified samples; watch approval metrics.

Next: [homework](02_homework.ipynb)